# Topic: Transformer Architecture Core (Positional Encoding, Encoder vs. Decoder, Masked Self-Attention)

## Definition (30-second explanation)
*   **Analogy:** Imagine assembling a team to write a book. The **Encoder** acts like a research team reading and summarizing the entire library at once with zero blind spots (bidirectional). The **Decoder** acts like the lead author writing one page at a time, strictly forbidden from reading ahead (autoregressive / causal masking). 
*   Because the team reads all pages at once without sequential order, they need **Positional Encodings** (page numbers with unique time-stamp waves) stamped directly onto each paragraph so order is preserved.

## Why Interviewers Ask This
*   To test whether you understand the fundamental design choices separating BERT (Encoder-only), GPT (Decoder-only), and T5 (Encoder-Decoder).
*   To see if you understand why Transformers can process tokens in parallel during training without losing the sequential order of language.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Self-attention treats inputs as an unordered set (permutation invariant). Without explicit order signals, "Dog bites man" produces the exact same representations as "Man bites dog".
*   **The Mechanism:** 
    *   **Positional Encoding:** Adds sinusoidal wave frequencies ($\sin$ for even dimensions, $\cos$ for odd) directly to token embeddings, giving every position a unique geometric fingerprint that allows the model to learn relative distances.
    *   **Encoder Block:** Bidirectional Multi-Head Attention + Feed-Forward Network with Residual Connections and LayerNorm (`Add & Norm`) to extract full-context representations.
    *   **Decoder Block:** Masked Self-Attention (prevents looking at future tokens) + Cross-Attention (queries the Encoder output) + Feed-Forward Network.
*   **The Trade-off:** Sinusoidal encodings struggle to generalize to sequence lengths significantly longer than those seen during pre-training, driving modern LLMs toward learned relative encodings like RoPE (Rotary Position Embeddings) or ALiBi.

## When to Use
*   **Encoder-only (BERT, RoBERTa):** Embedding generation, classification, semantic search, NER, and feature extraction.
*   **Decoder-only (GPT, LLaMA, Mistral):** Autoregressive text generation, code generation, reasoning, and conversational agents.
*   **Encoder-Decoder (T5, BART):** Sequence-to-sequence transformation tasks like abstractive summarization, translation, and structured data-to-text.

## Advantages
*   **Bidirectional Context in Encoders:** Full visibility across past and future tokens yields rich representations for understanding tasks.
*   **Autoregressive Alignment in Decoders:** Strict causal masking guarantees that training dynamics mirror token-by-token text generation.

## Limitations
*   **Extrapolation Limit of Fixed Positional Encoding:** Sinusoidal encoding does not scale smoothly beyond the pre-training context length without fine-tuning or interpolation.
*   **Inference Latency in Decoders:** Decoders must still generate tokens sequentially during inference ($O(N)$ sequential steps), requiring KV caching to avoid redundant recomputations.

## Common Comparisons
*   **Encoder vs. Decoder:** Encoders use unmasked self-attention (bidirectional); Decoders use causal masked self-attention (left-to-right only).
*   **Sinusoidal vs. Learned vs. Rotary (RoPE):** Sinusoidal uses fixed mathematical formulas; Learned encodings train a lookup table per position index (cannot extrapolate); RoPE rotates Query/Key vectors in complex space and is the standard for modern LLMs.

## Common Interview Traps
*   **Adding vs. Concatenating Positional Encodings:** Thinking positional encodings are concatenated to embeddings (which would increase vector dimension). They are **added** element-wise directly to the input embeddings ($X + PE$).
*   **Assuming Modern LLMs are Encoder-Decoder:** Believing modern models like GPT-4 or LLaMA have an Encoder block. Modern flagship LLMs are almost exclusively **Decoder-only**.

## Python / SQL Syntax (if applicable)
*   *Instantiating standard Transformer components using high-level Keras/TensorFlow APIs:*
```python
import tensorflow as tf
from tensorflow.keras import layers

# 1. Inputs: Batch=2, Sequence Length=8, Embedding Dim=64
inputs = tf.random.normal([2, 8, 64])

# 2. Encoder Layer (Bidirectional Self-Attention + Residual Add & Norm)
encoder_attn = layers.MultiHeadAttention(num_heads=4, key_dim=16)
x_enc = encoder_attn(query=inputs, value=inputs, key=inputs) # Full visibility
x_enc = layers.LayerNormalization()(inputs + x_enc) # Residual connection

# 3. Decoder Layer (Causal Masked Self-Attention)
decoder_attn = layers.MultiHeadAttention(num_heads=4, key_dim=16)
# use_causal_mask=True automatically injects the lower-triangular -inf mask
x_dec = decoder_attn(query=inputs, value=inputs, key=inputs, use_causal_mask=True)
x_dec = layers.LayerNormalization()(inputs + x_dec)
```

## Important Formula (if applicable)
*   **Sinusoidal Positional Encoding:**
    $$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
    $$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
    *(where $pos$ is token position index, $i$ is the dimension index, and $d_{\text{model}}$ is total embedding size).*

## 45-Second Interview Answer
*   "The original Transformer core consists of an Encoder that builds bidirectional representations using standard self-attention, and a Decoder that generates tokens autoregressively using causal masked self-attention and cross-attention. Because self-attention is permutation-invariant, positional encodings are added element-wise to input embeddings using sine and cosine waves of varying frequencies, allowing the model to distinguish word order and relative distances. Modern NLP split this architecture: Encoder-only models like BERT power embedding and classification tasks, while Decoder-only models like LLaMA dominate text generation."